# 🔬 RT-DETR-S — Esophageal Cancer Detection
### 150 Epochs | Fast Training | Loss Curves | Confusion Matrix | Metrics Table | best.pt

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — Install
# ═══════════════════════════════════════════════════════════
import os
os.system('pip install ultralytics -q')
print('✅ Done')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — Imports
# ═══════════════════════════════════════════════════════════
import json, shutil, warnings, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict
import torch
from ultralytics import RTDETR
warnings.filterwarnings('ignore')
print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only — enable GPU!"}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — Config  ← change COCO_ROOT if needed
# ═══════════════════════════════════════════════════════════
COCO_ROOT  = '/kaggle/input/datasets/rugvedkulkarni250809/esophageal-balanced-coco/coco_balanced'
ANNOTS_DIR = COCO_ROOT + '/annotations'
OUTPUT_DIR = '/kaggle/working/rtdetr_output'
EPOCHS     = 150
IMG_SIZE   = 480    # reduced for speed
BATCH_SIZE = 4      # reduced for speed
PATIENCE   = 75
DEVICE     = 0 if torch.cuda.is_available() else 'cpu'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Auto-detect classes — keep only AP and MP, skip MP-AP
with open(f'{ANNOTS_DIR}/instances_train.json') as f: _d = json.load(f)
VALID_NAMES  = ['AP', 'MP']
valid_cats   = [c for c in _d['categories'] if c['name'] in VALID_NAMES]
valid_cats   = sorted(valid_cats, key=lambda x: x['id'])
CLASS_NAMES  = [c['name'] for c in valid_cats]
VALID_CAT_IDS = {c['id']: i for i, c in enumerate(valid_cats)}  # 0-indexed

print(f'✅ Config OK')
print(f'   COCO_ROOT     : {COCO_ROOT}')
print(f'   Classes       : {CLASS_NAMES}')
print(f'   VALID_CAT_IDS : {VALID_CAT_IDS}')
print(f'   Model         : rtdetr-s.pt (fast)')
print(f'   IMG_SIZE      : {IMG_SIZE}')
print(f'   BATCH_SIZE    : {BATCH_SIZE}')
print(f'   EPOCHS        : {EPOCHS}')
print(f'   PATIENCE      : {PATIENCE}')
print(f'   Device        : {DEVICE}')
print()
for sp in ['train', 'val', 'test']:
    with open(f'{ANNOTS_DIR}/instances_{sp}.json') as f: dd = json.load(f)
    valid_anns = [a for a in dd['annotations'] if a['category_id'] in VALID_CAT_IDS]
    nf = len(list((Path(COCO_ROOT)/sp).glob('*.jpg')))
    print(f'  ✅ {sp:5s} → {len(dd["images"])} images | {len(valid_anns)} valid anns (AP+MP) | {nf} files')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — Convert COCO → YOLO (only AP and MP)
# ═══════════════════════════════════════════════════════════
def coco_to_yolo(ann_file, img_dir, out_img_dir, out_lbl_dir):
    Path(out_img_dir).mkdir(parents=True, exist_ok=True)
    Path(out_lbl_dir).mkdir(parents=True, exist_ok=True)
    with open(ann_file) as f: coco = json.load(f)
    abi = defaultdict(list)
    for a in coco['annotations']:
        if a['category_id'] in VALID_CAT_IDS:  # skip MP-AP
            abi[a['image_id']].append(a)
    n_imgs = 0
    for img in coco['images']:
        src = Path(img_dir) / img['file_name']
        if not src.exists(): continue
        W, H  = img['width'], img['height']
        lines = []
        for a in abi[img['id']]:
            x, y, w, h = a['bbox']
            if w < 5 or h < 5: continue
            cx = (x + w/2) / W
            cy = (y + h/2) / H
            nw = w / W
            nh = h / H
            cls = VALID_CAT_IDS[a['category_id']]
            lines.append(f'{cls} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')
        if lines:
            shutil.copy2(src, Path(out_img_dir) / img['file_name'])
            (Path(out_lbl_dir) / (Path(img['file_name']).stem + '.txt')).write_text('\n'.join(lines))
            n_imgs += 1
    print(f'  ✅ Converted {n_imgs} images → {out_lbl_dir}')

YOLO_DIR = '/kaggle/working/rtdetr_yolo'
for sp, folder in [('train','train'), ('val','valid'), ('test','test')]:
    coco_to_yolo(
        f'{ANNOTS_DIR}/instances_{sp}.json',
        f'{COCO_ROOT}/{sp}',
        f'{YOLO_DIR}/{folder}/images',
        f'{YOLO_DIR}/{folder}/labels'
    )
    ni = len(list(Path(f'{YOLO_DIR}/{folder}/images').glob('*.*')))
    nl = len(list(Path(f'{YOLO_DIR}/{folder}/labels').glob('*.txt')))
    print(f'    {folder:5s} → {ni} images | {nl} labels')
print('\n✅ COCO → YOLO conversion complete')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — Write data.yaml
# ═══════════════════════════════════════════════════════════
data_yaml = '/kaggle/working/rtdetr_data.yaml'
cfg = {
    'path' : YOLO_DIR,
    'train': 'train/images',
    'val'  : 'valid/images',
    'test' : 'test/images',
    'nc'   : len(CLASS_NAMES),
    'names': CLASS_NAMES,
}
with open(data_yaml, 'w') as f: yaml.dump(cfg, f, default_flow_style=False)
print('✅ data.yaml written:')
print(open(data_yaml).read())

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — Train RT-DETR-S (small = fast)
# ═══════════════════════════════════════════════════════════
model = RTDETR('rtdetr-s.pt')   # small model — 3x faster than rtdetr-l

results = model.train(
    data        = data_yaml,
    epochs      = EPOCHS,
    imgsz       = IMG_SIZE,
    batch       = BATCH_SIZE,
    device      = DEVICE,
    patience    = PATIENCE,
    project     = OUTPUT_DIR,
    name        = 'train',
    save        = True,
    save_period = 25,
    plots       = True,
    val         = True,
    verbose     = True,
    exist_ok    = True,
    workers     = 2,
)

run_dir  = Path(OUTPUT_DIR) / 'train'
best_pt  = run_dir / 'weights' / 'best.pt'
csv_path = run_dir / 'results.csv'
print(f'\n✅ Training complete!')
print(f'   best.pt  → {best_pt}')
print(f'   CSV log  → {csv_path}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — Loss + mAP Curves
# ═══════════════════════════════════════════════════════════
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()
print('CSV columns:', df.columns.tolist())

train_lc = [c for c in df.columns if 'train' in c.lower() and 'loss' in c.lower()]
val_lc   = [c for c in df.columns if 'val'   in c.lower() and 'loss' in c.lower()]

# Loss curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('RT-DETR-S — Esophageal Cancer — Loss Curves (150 Epochs)', fontsize=14, fontweight='bold')
for ax, (key, title) in zip(axes, [('box_loss','Box Loss'),('cls_loss','Class Loss'),('giou_loss','GIoU Loss')]):
    tc = next((c for c in train_lc if key in c.lower()), None)
    vc = next((c for c in val_lc   if key in c.lower()), None)
    if tc: ax.plot(df.index+1, df[tc], label='Train', color='#2196F3', linewidth=1.8)
    if vc: ax.plot(df.index+1, df[vc], label='Val',   color='#F44336', linewidth=1.8)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(); ax.grid(True, alpha=0.3)
    if len(df) > 0: ax.set_xlim(1, len(df))
plt.tight_layout()
loss_path = os.path.join(OUTPUT_DIR, 'rtdetr_loss_curves.png')
plt.savefig(loss_path, dpi=150, bbox_inches='tight'); plt.show()
print(f'✅ Loss curves saved: {loss_path}')

# mAP curves
map_cols = [c for c in df.columns if 'map' in c.lower()]
if map_cols:
    fig2, ax2 = plt.subplots(figsize=(10, 5))
    colors = ['#4CAF50','#FF9800','#9C27B0','#00BCD4']
    for i, col in enumerate(map_cols):
        ax2.plot(df.index+1, df[col], label=col.strip(), color=colors[i%len(colors)], linewidth=1.8)
    ax2.set_title('RT-DETR-S mAP over Epochs', fontweight='bold')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('mAP')
    ax2.legend(); ax2.grid(True, alpha=0.3)
    if len(df) > 0: ax2.set_xlim(1, len(df))
    plt.tight_layout()
    map_path = os.path.join(OUTPUT_DIR, 'rtdetr_map_curves.png')
    plt.savefig(map_path, dpi=150, bbox_inches='tight'); plt.show()
    print(f'✅ mAP curves saved: {map_path}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8 — Validate with best.pt + Confusion Matrix
# ═══════════════════════════════════════════════════════════
best_model  = RTDETR(str(best_pt))
val_results = best_model.val(
    data     = data_yaml,
    imgsz    = IMG_SIZE,
    batch    = BATCH_SIZE,
    device   = DEVICE,
    conf     = 0.25,
    iou      = 0.5,
    plots    = True,
    project  = OUTPUT_DIR,
    name     = 'val_best',
    exist_ok = True,
    verbose  = True,
)

# Get confusion matrix
cm_array = None
try:
    cm_array = val_results.confusion_matrix.matrix
    print('✅ CM loaded from val_results')
except Exception:
    pass
if cm_array is None:
    npy = Path(OUTPUT_DIR) / 'val_best' / 'confusion_matrix.npy'
    if npy.exists():
        cm_array = np.load(str(npy))
        print(f'✅ CM loaded from {npy}')

assert cm_array is not None, '❌ Confusion matrix not found!'

nc = len(CLASS_NAMES)
if cm_array.shape[0] == nc + 1:
    cm_array = cm_array[:nc, :nc]

print(f'CM shape: {cm_array.shape}')
print(cm_array)

cm_n = cm_array.astype(float)
rs   = cm_n.sum(axis=1, keepdims=True); rs[rs==0]=1; cm_n/=rs

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('RT-DETR-S — Confusion Matrix — Esophageal Cancer (AP vs MP)', fontsize=13, fontweight='bold')
for ax, d, t, f in zip(axes, [cm_array.astype(int), cm_n], ['Raw Counts','Normalised'], ['d','.2f']):
    sns.heatmap(d, annot=True, fmt=f, cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=ax, linewidths=0.8, annot_kws={'size':13,'weight':'bold'})
    ax.set_title(t, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
cm_path = os.path.join(OUTPUT_DIR, 'rtdetr_confusion_matrix.png')
plt.savefig(cm_path, dpi=150, bbox_inches='tight'); plt.show()
print(f'✅ Confusion matrix saved: {cm_path}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 9 — Metrics Table
# ═══════════════════════════════════════════════════════════
def compute_metrics(cm, names):
    rows = []; total = cm.sum()
    for i, cls in enumerate(names[:cm.shape[0]]):
        TP=cm[i,i]; FP=cm[:,i].sum()-TP; FN=cm[i,:].sum()-TP; TN=total-TP-FP-FN
        P  = TP/(TP+FP) if TP+FP>0 else 0
        R  = TP/(TP+FN) if TP+FN>0 else 0
        F1 = 2*P*R/(P+R) if P+R>0 else 0
        Acc= (TP+TN)/total if total>0 else 0
        rows.append({'Class':cls,'TP':int(TP),'FP':int(FP),'FN':int(FN),'TN':int(TN),
                     'Precision':round(P,4),'Recall':round(R,4),
                     'F1-Score':round(F1,4),'Accuracy':round(Acc,4)})
    rows.append({'Class':'MACRO AVG','TP':'','FP':'','FN':'','TN':'',
                 'Precision':round(np.mean([r['Precision'] for r in rows]),4),
                 'Recall'   :round(np.mean([r['Recall']    for r in rows]),4),
                 'F1-Score' :round(np.mean([r['F1-Score']  for r in rows]),4),
                 'Accuracy' :round(np.mean([r['Accuracy']  for r in rows]),4)})
    return pd.DataFrame(rows)

mdf = compute_metrics(cm_array.astype(float), CLASS_NAMES)
print('='*65)
print('  RT-DETR-S — PER CLASS METRICS')
print('='*65)
print(mdf.to_string(index=False))
print('='*65)

fig, ax = plt.subplots(figsize=(15, max(3, len(mdf)*0.65+1.2)))
ax.axis('off')
tbl = ax.table(cellText=mdf.values.tolist(), colLabels=list(mdf.columns), cellLoc='center', loc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.2, 1.8)
for j in range(len(mdf.columns)):
    tbl[0,j].set_facecolor('#1565C0')
    tbl[0,j].set_text_props(color='white', fontweight='bold')
for i in range(1, len(mdf)):
    for j in range(len(mdf.columns)):
        tbl[i,j].set_facecolor('#F5F5F5' if i%2==0 else 'white')
last = len(mdf)
for j in range(len(mdf.columns)):
    tbl[last,j].set_facecolor('#BBDEFB')
    tbl[last,j].set_text_props(fontweight='bold')
ax.set_title('RT-DETR-S — Precision / Recall / F1 / Accuracy', fontsize=13, fontweight='bold', pad=14)
plt.tight_layout()
tbl_path = os.path.join(OUTPUT_DIR, 'rtdetr_metrics_table.png')
plt.savefig(tbl_path, dpi=150, bbox_inches='tight'); plt.show()
mdf.to_csv(os.path.join(OUTPUT_DIR, 'rtdetr_metrics.csv'), index=False)
print(f'✅ Metrics table saved: {tbl_path}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 10 — Final Summary
# ═══════════════════════════════════════════════════════════
print('='*60)
print('  RT-DETR-S — ALL OUTPUT FILES')
print('='*60)
for name, path in [
    ('best.pt',           str(best_pt)),
    ('Loss Curves PNG',   os.path.join(OUTPUT_DIR, 'rtdetr_loss_curves.png')),
    ('mAP Curves PNG',    os.path.join(OUTPUT_DIR, 'rtdetr_map_curves.png')),
    ('Confusion Matrix',  os.path.join(OUTPUT_DIR, 'rtdetr_confusion_matrix.png')),
    ('Metrics Table PNG', os.path.join(OUTPUT_DIR, 'rtdetr_metrics_table.png')),
    ('Metrics CSV',       os.path.join(OUTPUT_DIR, 'rtdetr_metrics.csv')),
]:
    mark = '✅' if os.path.exists(path) else '❌'
    print(f'  {mark}  {name:<22} → {path}')
print('\n🎉 RT-DETR Training Complete!')